In [1]:
import json # for saving setting files
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split #use this for splitting

PATH = "data/raw/ML-EdgeIIoT-dataset.csv"
SEED = 42
TEST_SIZE = 0.30 #this fraction of the rows will be held for testing, THIS VALUE IS FIXED
MIN_TEST_PER_TYPE = 100 #minimum rows from each type to be tested
LABEL, TYPE = "Attack_label", "Attack_type"

DROP = {
    "MITM-only quirk artifact" : ["frame.time", "tcp.srcport","tcp.seq"],
    "address / timestamps" : ["ip.src_host", "ip.dst_host","arp.src.proto_ipv4", "arp.dst.proto_ipv4","icmp.transmit_timestamp"],
    "ID-like columns:" : ["tcp.ack_raw", "tcp.checksum","icmp.checksum","udp.stream", "icmp.seq_le"], #we will test it again with this included
    "ports" : ["tcp.dstport","udp.port"] #this one too
}

In [2]:
df=pd.read_csv(PATH, low_memory=False) #puts dataset into table
print("Rows:", len(df), "Columns:",df.shape[1])
agree = ((df[LABEL]==1) == (df[TYPE] != "Normal")).all() #checks if labels and attack types are greeing with each other
assert agree,  "Label and type disagree" #error if not

Rows: 157800 Columns: 63


In [3]:
#here we decide what columns to use

feature_cols = [c for c in df.columns if c not in (LABEL, TYPE)] # keep all columns except label and type
reason = {}
for why, cols in DROP.items():
    for c in cols:
        if c in df.columns:
            reason[c] = why
        else:
            print("Warning: not in file: ", c) #column from drop is not in fuile

for c in feature_cols:
    if c not in reason and df[c].nunique(dropna=False) <= 1:
        reason[c] = "constant"
for c in feature_cols:
    if c not in reason and not pd.api.types.is_numeric_dtype(df[c]):
        reason[c] = "text (we're using numbers only)"

kept = [c for c in feature_cols if c not in reason]
assert LABEL not in kept and TYPE not in kept

X = df[kept].astype("float64")
assert np.isfinite(X.to_numpy()).all(), "NaN or inf kept in columns"

print("Kept", len(kept), "of", len(feature_cols))
print("Columns kept: ")
print(kept)



Kept 25 of 61
Columns kept: 
['arp.opcode', 'arp.hw.size', 'http.content_length', 'http.response', 'tcp.ack', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.flags', 'tcp.flags.ack', 'tcp.len', 'udp.time_delta', 'dns.qry.name', 'dns.qry.qu', 'dns.retransmission', 'dns.retransmit_request', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msgtype', 'mqtt.proto_len', 'mqtt.topic_len', 'mqtt.ver']


In [4]:
# summary of what columns have been dropped and why:
for why in sorted(set(reason.values())):
    cols = [c for c in feature_cols if reason.get(c) == why]
    print(why, "->", len(cols), "columns:", cols)


ID-like columns: -> 5 columns: ['icmp.checksum', 'icmp.seq_le', 'tcp.ack_raw', 'tcp.checksum', 'udp.stream']
MITM-only quirk artifact -> 3 columns: ['frame.time', 'tcp.seq', 'tcp.srcport']
address / timestamps -> 5 columns: ['ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.src.proto_ipv4', 'icmp.transmit_timestamp']
constant -> 8 columns: ['icmp.unused', 'http.tls_port', 'dns.qry.type', 'dns.retransmit_request_in', 'mqtt.msg_decoded_as', 'mbtcp.len', 'mbtcp.trans_id', 'mbtcp.unit_id']
ports -> 2 columns: ['tcp.dstport', 'udp.port']
text (we're using numbers only) -> 13 columns: ['http.file_data', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'tcp.options', 'tcp.payload', 'dns.qry.name.len', 'mqtt.conack.flags', 'mqtt.msg', 'mqtt.protoname', 'mqtt.topic']


In [5]:
#find repeated rows

key = pd.util.hash_pandas_object(df[kept +[LABEL]], index=False).to_numpy()

order = np.random.default_rng(SEED).permutation(len(df))
later_copy = pd.Series(key[order]).duplicated().to_numpy()
dup = np.zeros(len(df),dtype=bool)
dup[order[later_copy]] = True

print("Rows identical to another row in kept features(+ label):",dup.sum())
print(df.loc[dup, TYPE].value_counts())


Rows identical to another row in kept features(+ label): 124281
Attack_type
Normal                   20259
DDoS_ICMP                14089
DDoS_UDP                 13096
Uploading                10154
Password                  9953
XSS                       9932
SQL_injection             9726
DDoS_HTTP                 9423
Ransomware                6849
Port_Scanning             5675
Backdoor                  5675
DDoS_TCP                  4237
Vulnerability_scanner     3113
MITM                      1099
Fingerprinting            1001
Name: count, dtype: int64


In [6]:
#remove duplicate rows

pos = np.flatnonzero(~dup)
sub = df.iloc[pos]

fkey = pd.util.hash_pandas_object(sub[kept], index=False).to_numpy()
nlab = pd.Series(sub[LABEL].to_numpy()).groupby(fkey).transform("nunique").to_numpy()

print("Rows remaining after removing duplicates: ", len(pos))
print("Features patterns with BOTH labels: " ,len(set(fkey[nlab > 1])), "(", (nlab > 1).sum(), " rows)")

Rows remaining after removing duplicates:  33519
Features patterns with BOTH labels:  18 ( 36  rows)


In [7]:
#perform split

tr, te = train_test_split(
    pos, test_size = TEST_SIZE, random_state = SEED, shuffle=True,
    stratify=df[LABEL].to_numpy()[pos], # stratify by label as type has one class with only one entry, where this isn't possible
)

tr, te = np.sort(tr), np.sort(te)
print("Training rows:", len(tr), "Testing rows:", len(te))

Training rows: 23463 Testing rows: 10056


In [8]:
assert not (set(tr) & set(te)), "a row is in both sets"
assert len(tr) + len(te) == len(pos)
assert not (set(key[tr]) & set(key[te])), "identical vector and label in both sets"
share_tr, share_te = df.iloc[tr][LABEL].mean(), df.iloc[te][LABEL].mean()
assert abs(share_tr-share_te)<0.01, "attack share differs between train and test sets"

tab = pd.DataFrame({
    "raw": df[TYPE].value_counts(),
    "dups_removed":df.loc[dup,TYPE].value_counts(),
    "kept": df.iloc[pos][TYPE].value_counts(),
    "train":df.iloc[tr][TYPE].value_counts(),
    "test":df.iloc[te][TYPE].value_counts(),
}).fillna(0).astype(int)
display(tab)

gone = tab.index[tab["kept"] < 2].tolist()
print("Fewer than 2 rows left (cannot be reported per type):", gone)
small = tab.index[tab["test"] < MIN_TEST_PER_TYPE].tolist()
print("Fewer than", MIN_TEST_PER_TYPE, "test rows(per-type results will be unreliable):", small)
print("Attack share train:", round(share_tr, 3), " test:", round(share_te, 3))
print("Normal rows in test:", (df.iloc[te][LABEL] == 0).sum())


,raw,dups_removed,kept,train,test
Attack_type,,,,,
Backdoor,10195,5675,4520,3188,1332
DDoS_HTTP,10561,9423,1138,779,359
DDoS_ICMP,14090,14089,1,0,1
DDoS_TCP,10247,4237,6010,4253,1757
DDoS_UDP,14498,13096,1402,997,405
Fingerprinting,1001,1001,0,0,0
MITM,1214,1099,115,78,37
Normal,24301,20259,4042,2829,1213
Password,9989,9953,36,23,13


Fewer than 2 rows left (cannot be reported per type): ['DDoS_ICMP', 'Fingerprinting']
Fewer than 100 test rows(per-type results will be unreliable): ['DDoS_ICMP', 'Fingerprinting', 'MITM', 'Password', 'Uploading', 'XSS']
Attack share train: 0.879  test: 0.879
Normal rows in test: 1213


In [11]:
# save file split

import os
os.makedirs("results", exist_ok=True)
np.save("results/train_split_numeric.npy",tr)
np.save("results/test_split_numeric.npy", te)

with open ("results/kept_columns_numeric.json","w") as f:
    json.dump({"kept": kept, "dropped": reason}, f, indent=2)

print("Saved split files to results/")

Saved split files to results/
